# MAMA-MIA — 01: Dataset Exploration

Maqsad: MAMA-MIA-Lite datasetini tekshirish va keyingi preprocessing strategiyasini aniqlash.

Tekshiriladigan savollar:
1. Nechta holat (case) bor va ular qaysi manba to'plamlardan (I-SPY1 / I-SPY2 / NACT / DUKE)?
2. Hajmlar (volume) o'lchami, voksel spacing, yo'nalish — qanchalik bir xil?
3. Intensivlik taqsimoti manbalar bo'yicha qanchalik farq qiladi? (domen siljishining asosiy ko'rsatkichi)
4. Maskalar: o'sma hajmi, nechta kesimda o'sma bor?

**Kaggle sozlamasi:** Accelerator = None (CPU), Internet = On.
Bu notebook GPU kvotasini sarflamaydi.

## 1. Muhitni tayyorlash

In [ ]:
!pip install -q huggingface_hub nibabel

import os, re, json, collections
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "YongchengYAO/MAMA-MIA-Lite"

# /kaggle/working 20 GB bilan cheklangan — vaqtinchalik fayllarni /kaggle/temp ga yozamiz
WORK = "/kaggle/temp/mamamia" if os.path.isdir("/kaggle") else "./mamamia"
os.makedirs(WORK, exist_ok=True)
print("Ish papkasi:", WORK)

## 2. Repo tarkibi — yuklab olmasdan

20.3 GB ni yuklamaymiz. Avval fayllar ro'yxatini olib, nomlaridan manba to'plamni ajratamiz.

MAMA-MIA holat ID'lari manba to'plam prefiksini saqlaydi (`ISPY1_`, `ISPY2_`, `NACT_`, `DUKE_`),
ya'ni **markazlararo bo'linish uchun alohida metadata kerak emas**.

In [ ]:
api = HfApi()
files = api.list_repo_files(REPO_ID, repo_type="dataset")

images = sorted(f for f in files if f.startswith("Images/") and f.endswith(".nii.gz"))
masks  = sorted(f for f in files if f.startswith("Masks/")  and f.endswith(".nii.gz"))

print(f"Images: {len(images)}   Masks: {len(masks)}")

def case_id(path):
    return os.path.basename(path).replace(".nii.gz", "")

img_ids = {case_id(f) for f in images}
msk_ids = {case_id(f) for f in masks}
print("Faqat rasmda bor :", len(img_ids - msk_ids))
print("Faqat maskada bor:", len(msk_ids - img_ids))

print("\nNamuna nomlar:")
for f in images[:5]:
    print("  ", f)

In [ ]:
def source_of(cid):
    """Holat ID'sidan manba to'plam nomini ajratadi."""
    m = re.match(r"([A-Za-z]+)", cid)
    return m.group(1).upper() if m else "UNKNOWN"

sources = collections.Counter(source_of(c) for c in sorted(img_ids))

print("Manba to'plamlar bo'yicha taqsimot:")
total = sum(sources.values())
for s, n in sources.most_common():
    print(f"  {s:<10} {n:>5}  ({100*n/total:5.1f}%)")
print(f"  {'JAMI':<10} {total:>5}")

# --- MUHIM TEKSHIRUV ---------------------------------------------------
# Butun leave-one-source-out protokoli shu bo'linishga tayanadi.
if len(sources) < 2:
    print("\n[!] OGOHLANTIRISH: ID'lardan manba ajralmadi.")
    print("    Yuqoridagi namuna nomlarga qarang va source_of() ni moslang.")
    print("    Agar nomlarda manba umuman kodlanmagan bo'lsa, MAMA-MIA klinik")
    print("    jadvali (Synapse syn60868042) kerak bo'ladi — unda markaz/manba")
    print("    ustuni bor. Bu holda rejani shunga qarab o'zgartiramiz.")
elif min(sources.values()) < 50:
    print(f"\n[!] Eng kichik guruh atigi {min(sources.values())} holat —")
    print("    leave-one-source-out fold'i sifatida juda kichik bo'lishi mumkin.")
else:
    print(f"\n[ok] {len(sources)} ta manba topildi — leave-one-source-out mumkin.")

## 3. Namuna yuklab olish

Har bir manbadan bir nechta holat olamiz — geometriya va intensivlikni solishtirish uchun.

In [ ]:
N_PER_SOURCE = 3

by_source = collections.defaultdict(list)
for cid in sorted(img_ids):
    by_source[source_of(cid)].append(cid)

sample_ids = []
for s, ids in by_source.items():
    sample_ids += ids[:N_PER_SOURCE]

print(f"Yuklanadigan holatlar: {len(sample_ids)}")

local = {}
for cid in sample_ids:
    ip = hf_hub_download(REPO_ID, f"Images/{cid}.nii.gz", repo_type="dataset", local_dir=WORK)
    mp = hf_hub_download(REPO_ID, f"Masks/{cid}.nii.gz",  repo_type="dataset", local_dir=WORK)
    local[cid] = (ip, mp)
    print("  ok:", cid)

## 4. Geometriya va intensivlik tahlili

Bu jadval domen siljishining miqdoriy ko'rsatkichi — manbalar orasida spacing va
intensivlik qanchalik farq qilishini ko'rsatadi.

In [ ]:
rows = []
for cid, (ip, mp) in local.items():
    img = nib.load(ip)
    msk = nib.load(mp)
    a = img.get_fdata(dtype=np.float32)
    m = np.asarray(msk.dataobj)

    sx, sy, sz = img.header.get_zooms()[:3]
    tumor_vox = int((m > 0).sum())
    tumor_slices = int((m > 0).any(axis=(0, 1)).sum())

    rows.append({
        "case": cid,
        "source": source_of(cid),
        "shape": tuple(a.shape),
        "spacing": (round(sx, 2), round(sy, 2), round(sz, 2)),
        "orient": "".join(nib.aff2axcodes(img.affine)),
        "int_p1": round(float(np.percentile(a, 1)), 1),
        "int_p99": round(float(np.percentile(a, 99)), 1),
        "tumor_vox": tumor_vox,
        "tumor_slices": tumor_slices,
        "n_slices": a.shape[2],
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows).sort_values(["source", "case"])
    display(df)
except ImportError:
    for r in rows:
        print(r)

In [ ]:
# Manbalar bo'yicha umumlashtirish
import pandas as pd
df = pd.DataFrame(rows)

print("Intensivlik diapazoni (p99) manbalar bo'yicha:")
print(df.groupby("source")["int_p99"].describe()[["mean", "min", "max"]].round(1))

print("\nO'sma bor kesimlar ulushi:")
df["tumor_frac"] = df["tumor_slices"] / df["n_slices"]
print(df.groupby("source")["tumor_frac"].mean().round(3))

## 5. Vizual tekshiruv

Har bir manbadan bitta holat — o'sma markazidagi aksial kesim, maska konturi bilan.

In [ ]:
one_per_source = {}
for r in rows:
    one_per_source.setdefault(r["source"], r["case"])

n = len(one_per_source)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4.5))
if n == 1:
    axes = [axes]

for ax, (src, cid) in zip(axes, sorted(one_per_source.items())):
    ip, mp = local[cid]
    a = nib.load(ip).get_fdata(dtype=np.float32)
    m = np.asarray(nib.load(mp).dataobj)

    # o'sma eng ko'p bo'lgan kesim
    counts = (m > 0).sum(axis=(0, 1))
    z = int(np.argmax(counts))

    sl = a[:, :, z]
    lo, hi = np.percentile(sl, [1, 99])
    ax.imshow(np.rot90(np.clip(sl, lo, hi)), cmap="gray")
    ax.contour(np.rot90(m[:, :, z] > 0), levels=[0.5], colors="red", linewidths=1.2)
    ax.set_title(f"{src}\n{cid}  (z={z})", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Xulosa va keyingi qadam

Yuqoridagi natijalardan quyidagilarni aniqlaymiz:

- **Manba taqsimoti** → leave-one-source-out protokolida har bir fold hajmi
- **Spacing farqi** → resampling kerakmi, qaysi izotrop o'lchamga
- **Intensivlik farqi** → normalizatsiya/harmonizatsiya strategiyasi (domen-generalizatsiya hissasining asosi)
- **O'sma kesimlar ulushi** → 2D yondashuvda sinf nomutanosibligi qanchalik og'ir

Keyingi notebook (`02_preprocessing`) shu qarorlar asosida 20.3 GB ni
qayta ishlab, Kaggle Dataset sifatida saqlanadigan ixcham versiyaga aylantiradi.